## Vector stores and Retrievers

Langchain vector stores and retrievers abstraction are designed to support retrieval of data from (vector) databases and other stores for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

Content:
- Documents 
- Vector stores
- Retrievals

#### Documents
Lanchain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has 2 attributes:
1. **page_content** - a string representing the content.
2. **metadata** - a dict containing arbitrary metadata. The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. 

Note that an individual Document object often represents a chunk of a larger Document. 

In [1]:
from langchain_core.documents import Document

documets = [
    Document(page_content='Dogs are great companions, known for their loyalty and friendliness.',
             metadata={ 'source': 'mamal-pets-doc' }),
    Document(page_content='Cats are independant pets that often enjoy their own space.',
             metadata={ 'source': 'mamal-pets-doc' }),
    Document(page_content='Goldfish are popular pets for begginers, requiring relatively simple care.',
             metadata={ 'source': 'fish-pets-doc' }),
    Document(page_content='Parrots are intelligent birds capable of mimicing human speech.',
             metadata={ 'source': 'bird-pets-doc' }),
    Document(page_content='Rabbits are social animals that need plenty of space to top around.', 
             metadata={ 'source': 'mamal-pets-doc' })
]

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

HUGGINGFACE_TOKEN = os.environ['HF_TOKEN']
GROQ_API_KEY = os.environ['GROQ_API_KEY']

print(f'HUGGINGFACE_TOKEN: {HUGGINGFACE_TOKEN[:3]}**{HUGGINGFACE_TOKEN[-3:]}')
print(f'GROQ API KEY: {GROQ_API_KEY[:3]}**{GROQ_API_KEY[-3:]}')

HUGGINGFACE_TOKEN: hf_**qiW
GROQ API KEY: gsk**qNU


Creating the LLM 

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='Llama3-8b-8192', api_key=GROQ_API_KEY)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x1169abe00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x116c44c20>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

Create the embedding techniques

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
embeddings

/opt/homebrew/anaconda3/envs/langchain-chatbots/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

Lets import these documents into **ChromaDB**

In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents=documets, embedding=embeddings)
vectorstore

In [9]:
vectorstore.similarity_search_with_score(query='cat')

[(Document(id='d2d7ba33-69cf-4ab6-bbdf-0c282dc6f517', metadata={'source': 'mamal-pets-doc'}, page_content='Cats are independant pets that often enjoy their own space.'),
  0.909964919090271),
 (Document(id='da953706-0892-4516-8abb-39eb4bc85dbc', metadata={'source': 'mamal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='47235590-d5c9-43c3-a211-3c8d4624bfe9', metadata={'source': 'mamal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to top around.'),
  1.593578815460205),
 (Document(id='4970e55a-ce9b-4fff-9a15-baaaaae9632e', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicing human speech.'),
  1.6666381359100342)]

We can also use **async query** in the Vector datastore. 

In [10]:
await vectorstore.asimilarity_search_with_score(query='Cat')

[(Document(id='d2d7ba33-69cf-4ab6-bbdf-0c282dc6f517', metadata={'source': 'mamal-pets-doc'}, page_content='Cats are independant pets that often enjoy their own space.'),
  0.909964919090271),
 (Document(id='da953706-0892-4516-8abb-39eb4bc85dbc', metadata={'source': 'mamal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='47235590-d5c9-43c3-a211-3c8d4624bfe9', metadata={'source': 'mamal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to top around.'),
  1.593578815460205),
 (Document(id='4970e55a-ce9b-4fff-9a15-baaaaae9632e', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicing human speech.'),
  1.6666381359100342)]

## Retrievers

Langchain Vector store classes do NOT subclass Runnable, and cannot imediatelly be integrated into Langchain Expression Language chains. 

Langchain Retrievers are Runnables, so they implement a standard set of methods (e.g. synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create Runnable easily. We will build one around the `similarity_search` method.

In [11]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

# This lambda (vectorstore.similarity_search) will be applied to all the parameters
# from batch method ('cat', 'dog'). First it will execute similarity_search with 'cat' and will get top 2 results (because we bind k=2)
# Then will do the same with 'dog' 
retriever = RunnableLambda(vectorstore.similarity_search).bind(k=2)
retriever.batch(inputs=['cat', 'dog'])

[[Document(id='d2d7ba33-69cf-4ab6-bbdf-0c282dc6f517', metadata={'source': 'mamal-pets-doc'}, page_content='Cats are independant pets that often enjoy their own space.'),
  Document(id='da953706-0892-4516-8abb-39eb4bc85dbc', metadata={'source': 'mamal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='da953706-0892-4516-8abb-39eb4bc85dbc', metadata={'source': 'mamal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  Document(id='d2d7ba33-69cf-4ab6-bbdf-0c282dc6f517', metadata={'source': 'mamal-pets-doc'}, page_content='Cats are independant pets that often enjoy their own space.')]]

Vectorstores implement `as_retriever()` method that will generate a Retriever, specifically a VectorStoreRetriever. Those Retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [12]:
retriever = vectorstore.as_retriever(search_type='similarity',
                                     search_kwargs={
                                         'k': 1
                                     })

retriever.batch(inputs=['cat', 'dog'])

[[Document(id='d2d7ba33-69cf-4ab6-bbdf-0c282dc6f517', metadata={'source': 'mamal-pets-doc'}, page_content='Cats are independant pets that often enjoy their own space.')],
 [Document(id='da953706-0892-4516-8abb-39eb4bc85dbc', metadata={'source': 'mamal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

## Integrate Retriever into LCEL chain

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

template = """
Answer this question using only the provided context:
{question}

Context:
{context}
"""

promt = ChatPromptTemplate.from_messages(messages=[
    ('human', template)
])

retriever = vectorstore.as_retriever(search_type='similarity',
                                     search_kwargs={
                                         'k': 1
                                     })

rag_chain = {'context': retriever, 'question': RunnablePassthrough()} | promt | llm

response = rag_chain.invoke(input='Tell me about Dogs')
response

AIMessage(content='According to the provided context, dogs are great companions, known for their loyalty and friendliness.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 79, 'total_tokens': 99, 'completion_time': 0.016021751, 'prompt_time': 0.009636943, 'queue_time': 0.085118984, 'total_time': 0.025658694}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--049a104a-a979-4ae5-a7d9-5dc52b654e98-0', usage_metadata={'input_tokens': 79, 'output_tokens': 20, 'total_tokens': 99})